# YOLOv8 Red & Green Box Detection — Colab GPU Training

Trains YOLOv8n on the **WRO detection** Roboflow dataset (`greenbox`, `redbox`, `xparking`), then exports `best.pt` and `best.onnx` for the Raspberry Pi.

**Before running anything:** go to `Runtime > Change runtime type` and select **T4 GPU**.

Then run the cells top to bottom. Pick **one** of the two dataset options (A or B).

In [ ]:
# 1. Verify the GPU is active
!nvidia-smi

In [ ]:
# 2. Install YOLOv8 + Roboflow
%pip install -q ultralytics roboflow
!yolo version

## Option A (recommended): download the dataset straight from Roboflow

Get your API key from Roboflow: click your profile picture > **Settings > API Keys**, copy the **Private API Key**, and paste it below. The download takes about a minute — much faster than uploading 200+ MB from your laptop.

In [ ]:
ROBOFLOW_API_KEY = "PASTE_YOUR_API_KEY_HERE"

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("danish-cq5li").project("wro-detection-47xs2")
dataset = project.version(1).download("yolov8")
DATASET_DIR = dataset.location
print("Dataset downloaded to:", DATASET_DIR)

## Option B: upload the dataset as a zip from your laptop

Only use this if Option A doesn't work. First zip the dataset on your laptop (PowerShell):

```powershell
cd d:\Course\Robotic\aupp-robotic\Assignment2
Compress-Archive -Path "WRO detection.v1i.yolov8\*" -DestinationPath wro_dataset.zip
```

Then run the next cell and pick `wro_dataset.zip` when the file chooser appears (upload takes a while — the zip is ~200 MB).

In [ ]:
# Option B only — skip this cell if you used Option A
from google.colab import files
import zipfile, os

uploaded = files.upload()  # choose wro_dataset.zip
zip_name = next(iter(uploaded))
DATASET_DIR = "/content/wro_dataset"
with zipfile.ZipFile(zip_name) as z:
    z.extractall(DATASET_DIR)
print("Dataset extracted to:", DATASET_DIR)

In [ ]:
# 3. Point data.yaml at the right folder (fixes Windows/relative paths from either option)
import yaml, os

data_yaml = os.path.join(DATASET_DIR, "data.yaml")
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)

cfg["path"] = DATASET_DIR
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
cfg["test"] = "test/images"

with open(data_yaml, "w") as f:
    yaml.safe_dump(cfg, f)

print(open(data_yaml).read())

In [ ]:
# 4. Train YOLOv8n on the GPU
# 10 epochs matches the assignment quick run (~15-20 min on a T4).
# For a stronger model, raise EPOCHS to 50-100 - early stopping (patience=20) stops it when it stops improving.
EPOCHS = 10

!yolo detect train model=yolov8n.pt data="{data_yaml}" imgsz=320 epochs={EPOCHS} batch=16 device=0 patience=20

In [ ]:
# 5. Look at the training curves and validation metrics
from IPython.display import Image, display
display(Image("runs/detect/train/results.png", width=900))

!yolo detect val model=runs/detect/train/weights/best.pt data="{data_yaml}" imgsz=320

In [ ]:
# 6. Export to ONNX for the Raspberry Pi (same settings as the assignment guide)
!yolo export model=runs/detect/train/weights/best.pt format=onnx imgsz=320 opset=12 dynamic=False simplify=False nms=True

In [ ]:
# 7. Download both weight files to your laptop
from google.colab import files
files.download("runs/detect/train/weights/best.pt")
files.download("runs/detect/train/weights/best.onnx")

## After downloading

Put both files in `d:\Course\Robotic\aupp-robotic\Assignment2`, then:

**Test on your laptop webcam** (PowerShell, venv activated):
```powershell
python detect_laptop.py --model best.pt --camera 0 --conf 0.5
```

**Deploy to the Raspberry Pi:**
```powershell
scp best.onnx aupp@pi-ip:/home/aupp/Documents/
scp app_pi_stream.py requirements-pi.txt aupp@pi-ip:/home/aupp/Documents/
```

Then on the Pi:
```bash
cd /home/aupp/Documents
python3 -m venv ~/yolo && source ~/yolo/bin/activate
pip install --upgrade pip && pip install -r requirements-pi.txt
python app_pi_stream.py --model best.onnx --camera 0 --host 0.0.0.0 --port 5000
```

Open `http://pi-ip:5000` in a browser to see the live detection stream.